# 03 · Features

**Goal:** turn the clean data into one model-ready table: one row per disruption, only features that are known
when the disruption starts (section F of notebook 02), the target and a train / validation / test label.

- **Input:** `data/processed/disruptions_clean.parquet` (from notebook 01)
- **Output:** `data/processed/model_table.parquet` (used by notebook 04)

Not done here: one-hot encoding and scaling. They happen in notebook 04 inside a scikit-learn pipeline, so that
they learn only from the training years.

**Rule used throughout:** every choice (which causes are rare, which line columns to keep) is **decided on the
training years only**, then **applied the same way to every row**.

## 1. Load the data and apply the scope rule

In [1]:
from pathlib import Path

import numpy as np
import pandas as pd

repo_root = Path.cwd().parent
processed_dir = repo_root / "data" / "processed"

THRESHOLD = 90      # minutes; fixed in notebook 02
MIN_TRAIN_ROWS = 30  # categories with fewer training rows are grouped

df = pd.read_parquet(processed_dir / "disruptions_clean.parquet")
df.shape

(37777, 15)

Strikes are out of scope (notebook 02, section C2): their length is known in advance. The rule is exactly the one
from notebook 02, including the restriction to the staff group, so `lightning strike` (weather) stays in.

In [2]:
is_strike = (df["cause_group"] == "staff") & df["cause_en"].str.contains("strike|staking", case=False, na=False)

cleaned = df.loc[~is_strike].copy()
cleaned["long"] = cleaned["duration_minutes"] >= THRESHOLD

assert len(cleaned) == 37305
print(f"Strike rows removed: {is_strike.sum()}, rows left: {len(cleaned):,}")

Strike rows removed: 472, rows left: 37,305


## 2. Train / validation / test split

In [3]:
cleaned["split"] = pd.cut(cleaned["year"], bins=[2018, 2023, 2024, 2025], labels=["train", "valid", "test"])

split_summary = cleaned.groupby("split", observed=False).agg(n=("long", "size"), long_rate=("long", "mean"))
split_summary.round(3)

,n,long_rate
split,,
train,24968,0.339
valid,5820,0.345
test,6517,0.345


**Why split by year and not randomly?** The model will be used on *future* disruptions. A random split would put
disruptions from the same week, the same incident (remember the back-to-back messages) and the same strike period in
both train and test, so the model would be tested on situations it has effectively already seen, and the score
would be too optimistic. Splitting by year tests exactly what matters: trained on 2019–2023, how well does it do
on a year it has never seen? 2024 is used to choose between models, 2025 only once at the end.

The long rate is almost the same in all three splits (34–35%), so the base rate does not shift.

## 3. Time features

In [4]:
cleaned["start_hour"] = cleaned["start_time"].dt.hour
cleaned["is_weekend"] = (cleaned["start_time"].dt.dayofweek >= 5).astype(int)

cleaned[["start_hour", "is_weekend"]].describe().loc[["min", "max", "mean"]]

,start_hour,is_weekend
min,0.00000,0.00000
max,23.00000,1.00000
mean,13.20646,0.20201


`start_hour` is the strongest time feature (notebook 02, D1). `is_weekend` is weak (D2) but cheap. Month and year are
not used.

## 4. Cause: group rare causes into `"other"`

In [5]:
train_cause_counts = cleaned.loc[cleaned["split"] == "train", "cause_en"].value_counts()
common_causes = train_cause_counts[train_cause_counts >= MIN_TRAIN_ROWS].index

# Applied to every row: rare in training OR never seen in training -> "other"
cleaned["cause"] = cleaned["cause_en"].where(cleaned["cause_en"].isin(common_causes), "other")

print("Causes kept:", len(common_causes))
(cleaned["cause"] == "other").groupby(cleaned["split"], observed=False).sum().rename("rows set to 'other'")

Causes kept: 48


split
train    262
valid     69
test      80
Name: rows set to 'other', dtype: int64

In [6]:
# Causes that appear in validation or test but never in training: handled automatically
unseen = ~cleaned["cause_en"].isin(train_cause_counts.index)
cleaned.loc[unseen].groupby(["split", "cause_en"], observed=True).size()

split  cause_en                             
valid  because of a change in the timetable     7
       deployment of security staff             2
       necessary adjustment to the timetable    3
       snow                                     1
test   because of a change in the timetable     2
       deployment of security staff             2
       emergency repairs                        2
dtype: int64

**Why is "rare" decided on the training years only?** In real use the model is built once and then meets new years.
If the rare list were based on all years, information from 2024–2025 would decide how the training data is
prepared: a small leak from the future. Deciding on train and applying the same list to all rows mimics real use.

**Unseen causes** (for example `snow` or `because of a change in the timetable`, which never occur in 2019–2023)
are not in the list, so they become `"other"` as well. The model has learned what "other" means, so it can still
make a sensible prediction for them.

The original `cause_en` column is kept unchanged, so before and after can always be compared.

## 5. Lines: one 0/1 column per line

A disruption can affect several lines, and the list of lines is sorted **alphabetically**, so "the first listed
line" would mean nothing:

In [7]:
line_lists = cleaned["rdt_lines"].dropna().str.split(", ")
multi_line = line_lists[line_lists.str.len() > 1]
print("Multi-line disruptions:", len(multi_line))
print("Share with an alphabetically sorted list:", multi_line.apply(lambda lines: lines == sorted(lines)).mean())

Multi-line disruptions: 12051
Share with an alphabetically sorted list: 1.0


Instead, every line gets its own 0/1 column (*multi-hot* encoding): 1 if the disruption affected that line.
Lines with fewer than 30 training rows are dropped, with the same "decide on train" rule as causes.
Column names are made safe for the models (letters, digits and underscores only).

In [8]:
line_dummies = cleaned["rdt_lines"].str.get_dummies(sep=", ")   # rows without a line get all zeros

train_line_counts = line_dummies.loc[cleaned["split"] == "train"].sum()
kept_lines = train_line_counts[train_line_counts >= MIN_TRAIN_ROWS].index
line_dummies = line_dummies[kept_lines]

line_dummies.columns = (
    "line_" + line_dummies.columns.str.lower().str.replace(r"[^a-z0-9]+", "_", regex=True).str.strip("_")
)
assert line_dummies.columns.is_unique

print("Distinct lines:", len(train_line_counts), "| kept:", len(kept_lines))
line_dummies.iloc[:3, :5]

Distinct lines: 111 | kept: 103


,line_s_hertogenbosch_eindhoven,line_s_hertogenbosch_nijmegen,line_s_hertogenbosch_tilburg,line_s_hertogenbosch_utrecht_centraal,line_aachen_hbf_heerlen
0,0,0,0,1,0
1,0,0,0,0,0
2,0,0,0,0,0


## 6. Number of lines affected

In [9]:
cleaned["n_lines"] = (cleaned["rdt_lines_id"].str.count(",") + 1).clip(upper=5)
print("Rows without a line:", cleaned["n_lines"].isna().sum())

cleaned["n_lines"] = cleaned["n_lines"].fillna(1).astype(int)
cleaned["n_lines"].value_counts().sort_index()

Rows without a line: 7


n_lines
1    25254
2     7482
3     2730
4      888
5      951
Name: count, dtype: int64

The 7 rows without a line get `n_lines = 1`: every disruption affects at least one line, the line is just not
recorded, and 1 is by far the most common value. Missing values are not allowed into the models.

## 7. Build and save the model table

In [10]:
id_and_target = ["rdt_id", "split", "long"]
features = ["cause", "cause_group", "start_hour", "is_weekend", "n_lines"]

model_table = pd.concat([cleaned[id_and_target + features], line_dummies], axis=1).reset_index(drop=True)

never_use = ["end_time", "duration_minutes", "statistical_cause_en", "statistical_cause_nl", "year", "start_time"]
assert not set(never_use) & set(model_table.columns)
assert model_table.isna().sum().sum() == 0
assert len(model_table) == 37305

print(model_table.shape)
model_table.head(3)

(37305, 111)


,rdt_id,split,long,cause,cause_group,start_hour,is_weekend,n_lines,line_s_hertogenbosch_eindhoven,line_s_hertogenbosch_nijmegen,...,line_roermond_sittard,line_roermond_venlo,line_roosendaal_vlissingen,line_rotterdam_centraal_schiphol_airport_hsl,line_rotterdam_centraal_utrecht_centraal,line_schiphol_airport_utrecht_centraal,line_schiphol_airport_zaandam,line_tiel_utrecht_centraal,line_utrecht_centraal_utrecht_maliebaan,line_winterswijk_zutphen
0,25236,train,False,damaged overhead wires,infrastructure,6,0,1,0,0,...,0,0,0,0,0,0,0,0,0,0
1,25237,train,True,other,external,7,0,3,0,0,...,0,0,0,0,0,0,0,0,0,0
2,25238,train,False,signal failure,infrastructure,8,0,1,0,0,...,0,0,0,0,0,0,0,0,0,0


In [11]:
model_table.to_parquet(processed_dir / "model_table.parquet", index=False)

check = pd.read_parquet(processed_dir / "model_table.parquet")
print("Saved:", check.shape)
check.dtypes.value_counts()

Saved: (37305, 111)


int64       106
str           2
category      1
bool          1
int32         1
Name: count, dtype: int64

## 8. Stability check: do cause rates hold between years?

In [12]:
top_causes = train_cause_counts.head(10).index

stability = (
    cleaned.loc[cleaned["cause"].isin(top_causes) & cleaned["split"].isin(["train", "valid"])]
    .pivot_table(index="cause", columns="split", values="long", aggfunc="mean", observed=True)
    .assign(difference=lambda t: t["valid"] - t["train"])
    .round(3)
    .sort_values("difference")
)
stability

split,train,valid,difference
cause,,,
broken down train,0.115,0.110,-0.006
an emergency call,0.098,0.106,0.008
collision,0.548,0.556,0.008
damaged overhead wires,0.665,0.683,0.018
person on the railway track,0.026,0.059,0.033
stranded train,0.132,0.166,0.033
points failure,0.566,0.621,0.055
signal failure,0.569,0.643,0.074
damaged railway bridge,0.378,0.504,0.126


Most common causes keep almost the same long rate in 2024 (within 1–3 points). Four move clearly upward:
`repair works` (+20 points), `damaged railway bridge` (+13), `signal failure` (+7) and `points failure` (+6).
A model that learned the 2019–2023 rates will **under-predict** long disruptions for these causes in 2024. This is
something to look for in the validation errors in notebook 04, and a reminder that the world the model learned
from keeps changing.

## Summary

- **37,305 disruptions**, split by year: train 2019–2023 (24,968), validation 2024 (5,820), test 2025 (6,517),
  each about 34–35% long.
- **Features (all known at the start):** `cause` (common causes, the rest `"other"`), `cause_group`, `start_hour`,
  `is_weekend`, `n_lines`, and one 0/1 column per line.
- **Rare causes and lines** were chosen on the training years only and the same rule was applied to every row;
  causes never seen in training become `"other"` automatically.
- **Not in the table:** anything only known after the disruption ends, plus `year`, `start_time` and the
  statistical cause (see notebook 02, section F). `rdt_id` is kept only as an identifier.
- **Watch in notebook 04:** some causes became more often long in 2024 (repair works, bridges, signals, points).
- Saved to `data/processed/model_table.parquet` for notebook 04.